# ਪਾਠ 13 - ਏਜੰਟ ਯਾਦਦਾਸ਼ਤ


## ਸੈਟਅਪ

ਇਹ ਨੋਟਬੁੱਕ ਇਹ ਦਰਸਾਉਂਦਾ ਹੈ ਕਿ **Microsoft Agent Framework** (MAF) ਦੀ ਵਰਤੋਂ ਕਰਕੇ **ਸਥਾਈ ਮੈਮੋਰੀ** ਵਾਲਾ ਯਾਤਰਾ ਬੁਕਿੰਗ ਏਜੰਟ ਕਿਵੇਂ ਬਣਾਇਆ ਜਾ ਸਕਦਾ ਹੈ।

ਤੁਹਾਨੂੰ ਸਿੱਖਣ ਨੂੰ ਮਿਲੇਗਾ ਕਿ ਏਜੰਟ ਦੀ ਵੱਖ-ਵੱਖ ਕਿਸਮਾਂ ਦੀ ਮੈਮੋਰੀ — ਵਰਕਿੰਗ, ਛੋਟੇ ਸਮੇਂ ਦੀ, ਅਤੇ ਲੰਮੇ ਸਮੇਂ ਦੀ — ਕਿਵੇਂ ਗੱਲਬਾਤਾਂ ਦੌਰਾਨ ਜਾਣਕਾਰੀ ਨੂੰ ਰੱਖਣ ਅਤੇ ਵਰਤਣ 'ਤੇ ਪ੍ਰਭਾਵ ਪਾਉਂਦੀਆਂ ਹਨ।

**ਲੋੜੀਂਦੇ ਤੱਤ:**
- ਇੱਕ Microsoft Foundry ਪ੍ਰੋਜੈਕਟ ਜਿਸ ਵਿੱਚ ਚੈਟ ਮਾਡਲ ਤੈਅ ਕੀਤਾ ਹੋਵੇ (ਜਿਵੇਂ `gpt-5-mini`)।
- Azure CLI ਨਾਲ ਲੌਗ ਇਨ ਕੀਤਾ ਹੋਇਆ — ਆਪਣੇ ਟਰਮੀਨਲ ਵਿੱਚ `az login` ਚਲਾਓ।
- `AZURE_AI_PROJECT_ENDPOINT` — ਤੁਹਾਡੇ Microsoft Foundry ਪ੍ਰੋਜੈਕਟ ਦਾ ਏન્ડਪੋਇੰਟ।
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — ਤੁਹਾਡੇ ਤੈਅ ਕੀਤਾ ਮਾਡਲ ਦਾ ਨਾਮ।


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import json
import dotenv
from typing import Annotated
from datetime import datetime

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## ਏਜੰਟ ਮੈਮੋਰੀ ਦੇ ਪ੍ਰਕਾਰ

AI ਏਜੰਟ ਵੱਖ-ਵੱਖ ਕਿਸਮ ਦੀਆਂ ਮੈਮੋਰੀਆਂ ਤੋਂ ਲਾਭਾਂ ਉਠਾ ਸਕਦੇ ਹਨ, ਜੋ ਹਰ ਇੱਕ ਵਿਲੱਖਣ ਮਕਸਦ ਦੀ ਪੇਸ਼ਕਸ਼ ਕਰਦੀਆਂ ਹਨ:

### ਵਰਕਿੰਗ ਮੈਮੋਰੀ
ਗੱਲਬਾਤ ਦਾ ਧਾਗਾ ਖੁਦ — ਇੱਕਲ-session ਵਿੱਚ ਬਦਲੇ ਗਏ ਸੁਨੇਹੇ। ਏਜੰਟ ਇੱਕੋ ਧਾਗੇ ਵਿੱਚ ਪਹਿਲਾਂ ਦੇ ਸੁਨੇਹਿਆਂ ਨੂੰ ਵਾਪਸ ਸੰਦਰਭ ਦੇ ਸਕਦਾ ਹੈ ਤਾਂ ਜੋ ਇਕਤਾ ਬਰਕਰਾਰ ਰਹੇ। MAF ਵਿੱਚ ਇਹ **`agent.create_session()`** ਰਾਹੀਂ ਬਣਾਇਆ ਜਾਂਦਾ ਹੈ, ਜੋ ਇੱਕ `AgentSession` ਵਾਪਸ ਕਰਦਾ ਹੈ।

### ਸ਼ਾਰਟ-ਟਰਮ ਮੈਮੋਰੀ
ਜਾਣਕਾਰੀ ਜੋ ਕਿਸੇ ਕੰਮ ਜਾਂ ਸੈਸ਼ਨ ਦੇ ਸਮੇਂ ਲਈ ਬਰਕਰਾਰ ਰਹਿੰਦੀ ਹੈ ਪਰ ਸਥਾਈ ਤੌਰ 'ਤੇ ਸਟੋਰ ਨਹੀਂ ਹੁੰਦੀ। ਉਦਾਹਰਨ ਵਜੋਂ, ਏਜੰਟ ਕਈ-ਚਰਣਾਂ ਵਾਲੀ ਯੋਜਨਾ ਬਾਤ ਚੀਤ ਦੌਰਾਨ ਤੱਥ ਇਕੱਠੇ ਕਰ ਸਕਦਾ ਹੈ ਅਤੇ ਫਾਇਨਲ ਯਾਤਰਾ ਸੋਚਣ ਲਈ ਉਨ੍ਹਾਂ ਦੀ ਵਰਤੋਂ ਕਰਦਾ ਹੈ।

### ਲੰਬੇ ਸਮੇਂ ਦੀ ਮੈਮੋਰੀ
ਪਸੰਦਾਂ ਅਤੇ ਤੱਥ ਜੋ **ਸੈਸ਼ਨਾਂ ਤੋਂ ਬਾਹਰ** ਬਰਕਰਾਰ ਰਹਿੰਦੇ ਹਨ। ਵਾਪਸ ਆਉਣ ਵਾਲੇ ਉਪਭੋਗਤਾ ਨੂੰ ਆਪਣੀਆਂ ਖੁਰਾਕੀ ਪਾਬੰਦੀਆਂ ਜਾਂ ਯਾਤਰਾ ਅੰਦਾਜ਼ ਮੁੜ ਨਹੀਂ ਦੱਸਣੇ ਪੈਣਗੇ। ਲੰਬੇ ਸਮੇਂ ਦੀ ਮੈਮੋਰੀ ਆਮ ਤੌਰ 'ਤੇ ਕਿਸੇ ਬਾਹਰੀ ਸਟੋਰ — ਡੇਟਾਬੇਸ, ਫਾਇਲ, ਜਾਂ ਵੇਕਟਰ ਇੰਡੈਕਸ — ਦੁਆਰਾ ਸਮਰਥਿਤ ਹੁੰਦੀ ਹੈ ਅਤੇ ਟੂਲਾਂ ਰਾਹੀਂ ਏਜੰਟ ਨੂੰ ਦਿਖਾਈ ਜਾਂਦੀ ਹੈ।


## ਸੈਸ਼ਨਾਂ ਨਾਲ ਵਰਕਿੰਗ ਮੈਮੋਰੀ

ਮੈਮੋਰੀ ਦਾ ਸਭ ਤੋਂ ਸਧਾਰਣ ਰੂਪ ਗੱਲਬਾਤ ਸੈਸ਼ਨ ਹੈ। ਜਦੋਂ ਤੁਸੀਂ ਇੱਕੋ ਸੈਸ਼ਨ (`agent.create_session()` ਰਾਹੀਂ ਬਣਾਇਆ ਗਿਆ) ਨੂੰ ਲਗਾਤਾਰ `agent.run()` ਕਾਲਾਂ ਵਿੱਚ ਪਾਸ ਕਰਦੇ ਹੋ, ਤਾਂ ਏਜੰਟ ਉਸ ਗੱਲਬਾਤ ਦਾ ਪੂਰਾ ਇਤਿਹਾਸ ਵੇਖਦਾ ਹੈ ਅਤੇ ਪਹਿਲਾਂ ਦੀਆਂ ਜਾਣਕਾਰੀਆਂ ਨੂੰ ਯਾਦ ਕਰ ਸਕਦਾ ਹੈ।

ਚਲੋ ਇੱਕ ਟ੍ਰੈਵਲ ਏਜੰਟ ਬਣਾਈਏ ਅਤੇ ਵਰਕਿੰਗ ਮੈਮੋਰੀ ਦਾ ਪ੍ਰਦਰਸ਼ਨ ਕਰੀਏ।


In [ ]:
agent = client.as_agent(
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

ਏਜੰਟ ਨੇ ਬਜਟ ਸਹੀ ਤਰ੍ਹਾਂ ਯਾਦ ਕੀਤਾ ਕਿਉਂਕਿ ਦੋਹਾਂ ਸੁਨੇਹਿਆਂ ਵਿੱਚ ਇੱਕੋ ਸੈਸ਼ਨ ਸਾਂਝਾ ਹੈ। ਇਹ **ਵਰਕਿੰਗ ਮੈਮੋਰੀ** ਹੈ — ਇਹ ਸਿਰਫ ਸੈਸ਼ਨ ਦੀ ਉਮਰ ਲਈ ਮੌਜੂਦ ਰਹਿੰਦੀ ਹੈ।

### ਨਵੇਂ ਥਰੇਡ ਨਾਲ ਕੀ ਹੁੰਦਾ ਹੈ?

ਜੇ ਅਸੀਂ ਇੱਕ **ਨਵਾਂ** ਸੈਸ਼ਨ ਬਣਾਈਏ, ਤਾਂ ਏਜੰਟ ਨੂੰ ਪਹਿਲੀ ਗੱਲਬਾਤ ਦੀ ਕੋਈ ਯਾਦ ਨਹੀਂ ਰਹਿੰਦੀ:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## ਲੰਬੇ ਸਮੇਂ ਦੀ ਯਾਦ ਦੀ ਪੈਟਰਨ

ਉਪਭੋਗਤਾ ਦੀਆਂ ਪਸੰਦਾਂ ਨੂੰ **ਸੈਸ਼ਨਾਂ ਦੇ ਪਾਰ** ਯਾਦ ਰੱਖਣ ਲਈ, ਸਾਨੂੰ ਇੱਕ ਪਾਇਦਾਰ ਸਟੋਰ ਦੀ ਲੋੜ ਹੈ ਜੋ ਗੱਲਬਾਤ ਦੇ ਧਾਗੇ ਤੋਂ ਬਾਹਰ ਰਹਿੰਦਾ ਹੈ। ਏਜੰਟ ਇਸ ਸਟੋਰ ਤੱਕ ਪਹੁੰਚਣ ਲਈ **ਟੂਲਜ਼** ਦਾ ਉਪਯੋਗ ਕਰਦਾ ਹੈ — ਫੰਕਸ਼ਨਾਂ ਨੂੰ ਜਿਹੜੇ ਇਹ ਜਾਣਕਾਰੀ ਸੰਭਾਲਣ ਅਤੇ ਪ੍ਰਾਪਤ ਕਰਨ ਲਈ ਕਾਲ ਕਰ ਸਕਦਾ ਹੈ।

ਹੇਠਾਂ ਅਸੀਂ ਇੱਕ ਸਧਾਰਣ ਮੈਮੋਰੀ ਅੰਦਰਲੀ ਪਸੰਦ ਸਟੋਰ ਨੂੰ ਲਾਗੂ ਕਰਦੇ ਹਾਂ (ਉਤਪਾਦਨ ਵਿੱਚ ਤੁਸੀਂ ਇਸ ਨੂੰ ਡੇਟਾਬੇਸ ਜਾਂ ਵੇਕਟਰ ਇੰਡੈਕਸ ਨਾਲ ਬੈਕ ਕਰਦੇ) ਅਤੇ ਇਸਨੂੰ ਉਹ ਟੂਲਜ਼ ਵਜੋਂ ਪੇਸ਼ ਕਰਦੇ ਹਾਂ ਜੋ ਏਜੰਟ ਵਰਤ ਸਕਦਾ ਹੈ।

### ਢਾਂਚਾ
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### ਸਥਿਤੀ 1 — ਪਹਿਲੀ ਵਾਰੀ ਉਪਭੋਗਤਾ ਵਰ੍ਹੇਗੰਢ ਦੀ ਯਾਤਰਾ ਬੁੱਕ ਕਰਦਾ ਹੈ

ਸਾਰਾਹ ਪਹਿਲੀ ਵਾਰੀ ਆ ਰਹੀ ਹੈ। ਏਜੰਟ ਨੂੰ ਉਸ ਦੀਆਂ ਪਸੰਦਾਂ ਸੰਦਾਂ ਰਾਹੀਂ ਸੰਗ੍ਰਹਿਤ ਕਰਨੀ ਚਾਹੀਦੀ ਹੈ ਅਤੇ ਉਹਨਾਂ ਮੁਤਾਬਕ ਹੋਟਲਾਂ ਦੀ ਸਿਫਾਰਸ਼ ਕਰਨੀ ਚਾਹੀਦੀ ਹੈ।


In [ ]:
travel_agent = client.as_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### ਮਾਮਲਾ 2 — ਸਾਰਾਹ ਹਫ਼ਤਿਆਂ ਬਾਅਦ ਵਾਪਸ ਆਉਂਦੀ ਹੈ

ਸਾਰਾਹ ਇੱਕ **ਬਿਲਕੁਲ ਨਵਾਂ ਥਰੇਡ** ਸ਼ੁਰੂ ਕਰਦੀ ਹੈ (ਨਵਾਂ ਸੈਸ਼ਨ ਦਿਖਾਉਂਦੀ ਹੈ)। ਵਰਕਿੰਗ ਮੈਮੋਰੀ ਖਾਲੀ ਹੈ, ਪਰ ਲੰਬੇ ਸਮੇਂ ਵਾਲਾ ਪਸੰਦ ਸਟੋਰ ਉਸ ਦੀ ਜਾਣਕਾਰੀ ਰੱਖਦਾ ਹੈ। ਏਜੰਟ ਨੂੰ ਇਹ ਪ੍ਰਾਪਤ ਕਰਕੇ ਸਿਫਾਰਸ਼ਾਂ ਨੂੰ ਨਿੱਜੀਕਰਨ ਵਾਸਤੇ ਵਰਤਣਾ ਚਾਹੀਦਾ ਹੈ।


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## ਸੰਖੇਪ

ਇਸ ਪਾਠ ਵਿੱਚ ਤੁਸੀਂ ਤਿੰਨ ਕਿਸਮਾਂ ਦੀ ਏਜੰਟ ਮੈਮੋਰੀ ਅਤੇ ਮਾਈਕ੍ਰੋਸਾਫਟ ਏਜੰਟ ਫਰੇਮਵਰਕ ਨਾਲ ਉਨ੍ਹਾਂ ਨੂੰ ਲਾਗੂ ਕਰਨ ਦਾ ਤਰੀਕਾ ਸਿੱਖਿਆ:

| ਮੈਮੋਰੀ ਦੀ ਕਿਸਮ | MAF ਮਕੈਜ਼ਨ | ਜੀਵਨਕਾਲ |
|---|---|---|
| **ਵਰਕਿੰਗ** | `agent.create_session()` | ਇਕ ਵਿਅਕਤ ਗੱਲਬਾਤ |
| **ਛੋਟੀ ਮਿਆਦ ਦੀ** | ਇੱਕ ਧਾਗੇ ਵਿੱਚ ਇਕੱਠਾ ਕੀਤਾ ਸੰਦਰਭ | ਇਕ ਕੰਮ / ਸੈਸ਼ਨ |
| **ਲੰਬੀ ਮਿਆਦ ਦੀ** | ਬਾਹਰੀ ਸਟੋਰ ਜਿਸ ਤੱਕ ਪਹੁੰਚ `@tool` ਫੰਕਸ਼ਨਾਂ ਰਾਹੀਂ ਹੋੰਦੀ ਹੈ | ਸੈਸ਼ਨਾਂ ਵਿੱਚ |

### ਮੁਖ ਚੀਜ਼ਾਂ
1. **`agent.create_session()`** ਵਰਕਿੰਗ ਮੈਮੋਰੀ ਮੁਹੱਈਆ ਕਰਦਾ ਹੈ — ਏਜੰਟ ਸੈਸ਼ਨ ਵਿੱਚ ਪੂਰੀ ਗੱਲਬਾਤ ਇਤਿਹਾਸ ਨੂੰ ਵੇਖਦਾ ਹੈ।
2. **ਨਵੇਂ ਸੈਸ਼ਨ ਸੰਦਰਭ ਖੋ ਦੇਂਦੇ ਹਨ** — ਲੰਬੀ ਮਿਆਦ ਦੀ ਮੈਮੋਰੀ ਦੇ ਬਿਨਾ ਏਜੰਟ ਪਿਛਲੇ ਗੱਲਬਾਤਾਂ ਨੂੰ ਯਾਦ ਨਹੀਂ ਰੱਖ ਸਕਦਾ।
3. **`@tool` ਫੰਕਸ਼ਨ** ਖਾਈ ਪੂਰੀ ਕਰਦੇ ਹਨ — ਇਹਨਾਂ ਰਾਹੀਂ ਏਜੰਟ ਜਾਣਕਾਰੀ ਨੂੰ ਸਥਾਈ ਸਟੋਰ ਵਿੱਚ ਸੇਵ ਅਤੇ ਪ੍ਰਾਪਤ ਕਰ ਸਕਦਾ ਹੈ।
4. **ਵੈਯਕਤੀਕਰਨ ਸਮੇਂ ਦੇ ਨਾਲ ਬਿਹਤਰ ਹੁੰਦਾ ਹੈ** — ਜਿੰਨੀ ਵਡੀ ਮਾਤਰਾ ਵਿੱਚ ਪ੍ਰੇਫਰੈਂਸ ਸਟੋਰ ਕੀਤੀਆਂ ਜਾਣ, ਏਜੰਟ ਦੀ ਸਿਫਾਰਿਸ਼ਾਂ ਅਧਿਕ ਪ੍ਰਭਾਵਸ਼ਾਲੀ ਹੁੰਦੀਆਂ ਹਨ।

### ਅਸਲੀ ਦੁਨਿਆ ਦੇ ਪ੍ਰਯੋਗ
- **ਗਾਹਕ ਸੇਵਾ**: ਗਾਹਕ ਇਤਿਹਾਸ ਅਤੇ ਪਸੰਦਾਂ ਨੂੰ ਯਾਦ ਰੱਖੋ
- **ਵੈਯਕਤੀਗਤ ਸਹਾਇਕ**: ਦਿਨਾਂ ਜਾਂ ਹਫ਼ਤਿਆਂ ਵਿੱਚ ਸੰਦਰਭ ਬਣਾਇਆ ਰੱਖੋ
- **ਸਿਹਤ ਸੇਵਾ**: ਮਰੀਜ਼ ਦੀ ਜਾਣਕਾਰੀ ਅਤੇ ਪ੍ਰੇਫਰੈਂਸ ਟਰੈਕ ਕਰੋ
- **ਈ-ਕੌਮੇਰਸ**: ਇਤਿਹਾਸ ਅਧਾਰਤ ਵੈਯਕਤੀਗਤ ਖਰੀਦਦਾਰੀ

### ਅਗਲੇ ਕਦਮ
- ਮੈਮੋਰੀ ਡਿਕਸ਼ਨਰੀ ਦੀ ਥਾਂ ਇੱਕ ਡੇਟਾਬੇਸ ਜਾਂ ਵੈਕਟਰ ਸਟੋਰ (ਜਿਵੇਂ Azure AI Search) ਵਰਗਾ ਵਿਕਲਪ ਰੱਖੋ
- ਸਮੇਂ-ਸੰਵੇਦਨਸ਼ੀਲ ਜਾਣਕਾਰੀ ਲਈ ਮੈਮੋਰੀ ਦੀ ਮਿਆਦ ਨਿਰਧਾਰਿਤ ਕਰੋ
- ਸਾਂਝੀ ਮੈਮੋਰੀ ਨਾਲ ਬਹੁ ਏਜੰਟ ਪ੍ਰਣਾਲੀ ਬਣਾਓ
- ਗਿਆਨ-ਗ੍ਰਾਫ-ਸਹਾਇਤ ਮੈਮੋਰੀ ਲਈ Cognee ਨੋਟਬੁੱਕ ਦੀ ਖੋਜ ਕਰੋ


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ਅਸਵੀਕਾਰੋਪਣ**:
ਇਸ ਦਸਤਾਵੇਜ਼ ਦਾ ਅਨੁਵਾਦ ਏਆਈ ਅਨੁਵਾਦ ਸੇਵਾ [Co-op Translator](https://github.com/Azure/co-op-translator) ਦੀ ਵਰਤੋਂ ਕਰਕੇ ਕੀਤਾ ਗਿਆ ਹੈ। ਜਦੋਂ ਕਿ ਅਸੀਂ ਸਹੀਤਾਵਾਂ ਲਈ ਯਤਨਸ਼ੀਲ ਹਾਂ, ਕਿਰਪਾ ਕਰਕੇ ਧਿਆਨ ਰੱਖੋ ਕਿ ਸਵੈਚਾਲਿਤ ਅਨੁਵਾਦਾਂ ਵਿੱਚ ਗਲਤੀਆਂ ਜਾਂ ਅਸਮੱਤਿਆਵਾਂ ਹੋ ਸਕਦੀਆਂ ਹਨ। ਮੂਲ ਦਸਤਾਵੇਜ਼ ਆਪਣੀ ਮੂਲ ਭਾਸ਼ਾ ਵਿੱਚ ਅਧਿਕਾਰਕ ਸਰੋਤ ਮੰਨਿਆ ਜਾਣਾ ਚਾਹੀਦਾ ਹੈ। ਜਰੂਰੀ ਜਾਣਕਾਰੀ ਲਈ, ਪੇਸ਼ੇਵਰ ਮਨੁੱਖੀ ਅਨੁਵਾਦ ਦੀ ਸਿਫ਼ਾਰਸ਼ ਕੀਤੀ ਜਾਂਦੀ ਹੈ। ਅਸੀਂ ਇਸ ਅਨੁਵਾਦ ਦੇ ਉਪਯੋਗ ਤੋਂ ਪੈਦਾ ਹੋਣ ਵਾਲੀਆਂ ਕਿਸੇ ਵੀ ਗਲਤਫਹਿਮੀਆਂ ਜਾਂ ਗਲਤ ਵਿਆਖਿਆਵਾਂ ਲਈ ਜਵਾਬਦੇਹ ਨਹੀਂ ਹਾਂ।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
